In [3]:
import numpy as np

np.random.seed(2026)
n = 10_000

diem_tin_dung = np.random.normal(680, 80, n).clip(300, 850)
thu_nhap      = np.random.lognormal(3, 0.5, n) * 1e6
du_no         = np.random.uniform(0, 500e6, n)
so_thang_vay  = np.random.choice([12,24,36,48,60,120,240,360], n)
lai_suat      = np.random.uniform(6, 20, n)
nhom_no       = np.random.choice([1,2,3,4,5], n,p=[0.5,0.25,0.15,0.07,0.03])

data = np.column_stack([diem_tin_dung, thu_nhap, du_no])

#Module 01 — Làm sạch
#Phát hiện và clip outlier (±3 std) cho diem_tin_dung, thu_nhap, du_no
mean_data = np.mean(data, axis = 0)
std_data = np.std(data, axis = 0)
lower = mean_data - 3*std_data
upper = mean_data + 3*std_data

outlier_mask = (data < lower) | (data > upper)
so_dong_bi_clip = np.sum(np.any(outlier_mask, axis=1))
#print(f"Số dòng có outlier bị clip: {so_dong_bi_clip}")

data_clipped = np.clip(data, lower, upper)

#print(data_clipped)

#Kiểm tra giá trị bất hợp lý: diem_tin_dung < 300 hoặc du_no < 0
bat_hop_ly = np.sum(data_clipped[:, 0] < 300) + np.sum(data_clipped[:, 2] < 0)
#print(f"Giá trị bất hợp lý sau clip: {bat_hop_ly}")

##Module 02 - Feature engineering
diem_clean   = data_clipped[:, 0]
thu_nhap_clean = data_clipped[:, 1]
du_no_clean  = data_clipped[:, 2]

DTI = du_no_clean / (thu_nhap_clean * so_thang_vay)
tong_lai = du_no_clean * (lai_suat/ 100) * (so_thang_vay / 12)

#Nhóm rủi ro: Low (diem >= 700 và DTI < 0.3) / High (diem < 600 hoặc DTI > 0.5) / Medium (còn lại)
condition = [(diem_clean >= 700) & (DTI < 0.3), (diem_clean < 600) | (DTI > 0.5)]
phan_loai = ['Low',                                 'High']
risk_group = np.select(condition, phan_loai, default = 'Medium')

# In phân phối nhóm rủi ro
for nhom in ["Low", "Medium", "High"]:
    so_luong = np.sum(risk_group == nhom)
    print(f"{nhom}: {so_luong} ({so_luong/n*100:.1f}%)")


# Module 3 — Phân tích thống kê
features = np.column_stack([diem_clean, thu_nhap_clean, du_no_clean, DTI])

# Ma trận tương quan — tự tính
Z = (features - np.mean(features, axis=0)) / np.std(features, axis=0, ddof=1)
corr_matrix = Z.T @ Z / (n - 1)

# Kiểm tra
print(np.allclose(corr_matrix, np.corrcoef(features.T), atol=1e-5))

# Thống kê theo nhóm rủi ro
for nhom in ["Low", "Medium", "High"]:
    mask = risk_group == nhom
    print(f"\n--- {nhom} ---")
    print(f"Điểm  : mean={diem_clean[mask].mean():.1f}, std={diem_clean[mask].std():.1f}")
    print(f"DTI   : P25={np.percentile(DTI[mask],25):.3f}, P75={np.percentile(DTI[mask],75):.3f}")

# Top 100 hồ sơ rủi ro nhất
# rủi ro = điểm thấp + DTI cao → dùng điểm tổng hợp
diem_rui_ro = -diem_clean + DTI * 100  # điểm cao = rủi ro cao
top100_idx = np.argsort(diem_rui_ro)[-100:]
print(f"\nTop 100 rủi ro — điểm TB: {diem_clean[top100_idx].mean():.1f}")
print(f"Top 100 rủi ro — DTI TB : {DTI[top100_idx].mean():.3f}")

# Báo cáo tổng hợp
print("=" * 50)
print("BÁO CÁO DANH MỤC TÍN DỤNG — 10,000 HỒ SƠ")
print("=" * 50)
print(f"1. Tổng hồ sơ              : {n:,}")
print(f"2. Hồ sơ có outlier        : {so_dong_bi_clip:,}")
print(f"3. Điểm TB toàn danh mục   : {diem_clean.mean():.1f}")
print(f"4. Điểm trung vị           : {np.median(diem_clean):.1f}")
print(f"5. DTI trung bình          : {DTI.mean():.3f}")
print(f"6. Tổng lãi TB (triệu đồng): {tong_lai.mean()/1e6:.1f}")
print(f"7. Nhóm Low Risk           : {np.sum(risk_group=='Low'):,} ({np.sum(risk_group=='Low')/n*100:.1f}%)")
print(f"8. Nhóm High Risk          : {np.sum(risk_group=='High'):,} ({np.sum(risk_group=='High')/n*100:.1f}%)")
print(f"9. Điểm TB nhóm High Risk  : {diem_clean[risk_group=='High'].mean():.1f}")
print(f"10. DTI TB nhóm High Risk  : {DTI[risk_group=='High'].mean():.3f}")

Low: 2689 (26.9%)
Medium: 3933 (39.3%)
High: 3378 (33.8%)
True

--- Low ---
Điểm  : mean=756.5, std=41.3
DTI   : P25=0.034, P75=0.153

--- Medium ---
Điểm  : mean=668.1, std=46.9
DTI   : P25=0.043, P75=0.315

--- High ---
Điểm  : mean=636.0, std=89.6
DTI   : P25=0.201, P75=0.996

Top 100 rủi ro — điểm TB: 600.1
Top 100 rủi ro — DTI TB : 3.115
BÁO CÁO DANH MỤC TÍN DỤNG — 10,000 HỒ SƠ
1. Tổng hồ sơ              : 10,000
2. Hồ sơ có outlier        : 167
3. Điểm TB toàn danh mục   : 681.0
4. Điểm trung vị           : 680.9
5. DTI trung bình          : 0.354
6. Tổng lãi TB (triệu đồng): 301.9
7. Nhóm Low Risk           : 2,689 (26.9%)
8. Nhóm High Risk          : 3,378 (33.8%)
9. Điểm TB nhóm High Risk  : 636.0
10. DTI TB nhóm High Risk  : 0.760
